In [0]:
dbutils.widgets.text("p_batch_id", "", "Batch ID")

v_batch_id = dbutils.widgets.get("p_batch_id").strip()

if not v_batch_id:
    raise ValueError("Enter a batch ID")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/04.gold-helpers

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
constructors_silver_path = "s3://bizf1project123/Silver/Aaisha/incremental-load/constructors"

nationality_region_gold_path = "s3://bizf1project123/Gold/Aaisha/ref_nationality_region"

gold_path = "s3://bizf1project123/Gold/Aaisha/incremental-load/dim_constructors"

In [0]:
constructors_df = (
    spark.read.format("delta")
        .load(constructors_silver_path)
        .filter(F.col("batch_id") == v_batch_id)
)

print("Constructors rows:", constructors_df.count())

In [0]:
ref_nationality_region_df = (
    spark.read.format("delta")
        .load(nationality_region_gold_path)
)

print("Nationality reference rows:", ref_nationality_region_df.count())

In [0]:
display(constructors_df)

In [0]:
display(ref_nationality_region_df)

In [0]:
dim_constructors_df = (
    constructors_df.alias("c")
        .join(
            ref_nationality_region_df.alias("r"),
            F.col("c.nationality") == F.col("r.nationality"),
            "left"
        )
        .select(
            F.col("c.constructor_id"),
            F.col("c.constructor_name"),
            F.col("c.nationality"),
            F.col("r.region").alias("nationality_region"),
            F.col("c.batch_id")
        )
)

In [0]:
print("Joined rows:", dim_constructors_df.count())

display(dim_constructors_df)

In [0]:
dim_constructors_df = (
    dim_constructors_df
        .withColumn("created_timestamp", F.current_timestamp())
        .withColumn("updated_timestamp", F.current_timestamp())
)

In [0]:
display(dim_constructors_df)

In [0]:
if not DeltaTable.isDeltaTable(spark, gold_path):
    (
        dim_constructors_df.write
            .format("delta")
            .mode("overwrite")
            .save(gold_path)
    )
else:
    gold_delta = DeltaTable.forPath(spark, gold_path)

    (
        gold_delta.alias("t")
            .merge(
                dim_constructors_df.alias("s"),
                "t.constructor_id = s.constructor_id"
            )
            .whenMatchedUpdate(
                condition="s.batch_id >= t.batch_id",
                set={
                    "constructor_name": "s.constructor_name",
                    "nationality": "s.nationality",
                    "nationality_region": "s.nationality_region",
                    "batch_id": "s.batch_id",
                    "updated_timestamp": "s.updated_timestamp"
                }
            )
            .whenNotMatchedInsertAll()
            .execute()
    )

In [0]:
dim_constructors_gold_df = (
    spark.read.format("delta").load(gold_path)
)

print("Total Gold Constructors rows:", dim_constructors_gold_df.count())

display(dim_constructors_gold_df)